# Partycjonowanie

Milestone 2 — trzy układy danych i pięć konfiguracji testu.

| Konfiguracja | Układ | Zakres |
|---|---|---|
| A | Jeden plik Parquet | Wybrane stacje |
| B | Niewiele grup stacji | Wybrane stacje + właściwe grupy |
| B bez filtra | Te same grupy | Cały zbiór, bez WHERE |
| C | Jedna partycja na stację | Wybrane stacje |
| C bez filtra | Te same partycje stacji | Cały zbiór, bez WHERE |

Dodatkowy scenariusz „wszystkie stacje” porównuje wszystkie konfiguracje bez ograniczenia zbioru.
W pozostałych scenariuszach **bez filtra oznacza inny zakres danych**: nie liczymy jego zysku
względem filtrowanego A. B bez filtra porównujemy z C bez filtra.

Skrypt zawiera całą logikę. Notebook wywołuje funkcje i wyświetla wyniki. Nie publikuje danych
na miejsce dotychczasowego SILVER — każdy eksperyment ma własny katalog.

## 1. Ustawienia

Domyślnie B ma 8 grup, a C tyle partycji, ile jest stacji. Dla 75 stacji daje to około
9–10 stacji w grupie. Algorytm wyrównuje liczbę wierszy, a nie rozmiar skompresowanych plików.
8 grup to wariant eksperymentu, nie gwarantowane optimum. Można przetestować np. 4, 8 i 16
w osobnych uruchomieniach, nie wybierając najlepszego wyniku z pojedynczego pomiaru.

In [ ]:
from pathlib import Path
from uuid import uuid4
import inspect
import runpy
import polars as pl
from IPython.display import display, Markdown, Image

PROJECT_ROOT = None  # np. r"D:\Projekty\gios-bigdata"
AUTO_BUILD_FEATURES = True
GROUP_COUNT = 8
REPEATS = 20
WARMUPS = 3
THREADS = None
ROW_GROUP_SIZE = 122880
WRITE_STATISTICS = False
READER_MODE = "all_files_list"  # lub "glob"; identyczny tryb B i C
SMALL_DIFFERENCE_PERCENT = 5.0


def find_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts/partition_benchmark.py").is_file():
            return candidate
    raise FileNotFoundError("Ustaw PROJECT_ROOT na katalog projektu zawierający scripts/")


ROOT = find_root(PROJECT_ROOT if PROJECT_ROOT is not None else Path.cwd())
SILVER = ROOT / "data/silver"
SOURCE = SILVER / "pm25_pogoda_2024.parquet"
old_ctx = globals().pop("ctx", None)
if isinstance(old_ctx, dict) and old_ctx.get("con") is not None:
    old_ctx["con"].close()
del old_ctx
print("Projekt:", ROOT)
print("Źródło:", SOURCE)

## 2. Dane wejściowe

Jeżeli tabela cech nie istnieje, tworzymy ją funkcjami z `silver_features.py`.
Wymaga to danych z etapu 04 i metadanych stacji BRONZE. Istniejącego źródła nie nadpisujemy.
Zapis A/B/C wczytuje całą tabelę do RAM.

In [ ]:
if not SOURCE.is_file():
    if SOURCE.exists():
        raise ValueError(f"Ścieżka nie jest plikiem: {SOURCE}")
    if not AUTO_BUILD_FEATURES:
        raise FileNotFoundError(f"Uruchom notebook 05. Brak {SOURCE}")
    feature_script = ROOT / "scripts/silver_features.py"
    inputs = [SILVER / "pomiary_2024.parquet", SILVER / "openmeteo_2024.parquet",
              ROOT / "data/bronze/stacje.parquet"]
    missing = [str(p) for p in [feature_script, *inputs] if not p.is_file()]
    if missing:
        raise FileNotFoundError("Brakuje danych wcześniejszych etapów:\n" + "\n".join(missing))
    api = runpy.run_path(str(feature_script))
    features = api["add_features"](api["build_base"](
        pl.read_parquet(inputs[0]), pl.read_parquet(inputs[1]), pl.read_parquet(inputs[2])))
    if features.is_empty():
        raise ValueError("Pusta tabela cech")
    SILVER.mkdir(parents=True, exist_ok=True)
    temporary = SOURCE.with_name(f".{SOURCE.name}.{uuid4().hex}.tmp")
    try:
        features.write_parquet(temporary, compression="snappy")
        if SOURCE.exists():
            raise FileExistsError("Inny proces utworzył źródło. Ponów uruchomienie.")
        temporary.rename(SOURCE)
    finally:
        temporary.unlink(missing_ok=True)
    print("Utworzono tabelę cech:", features.shape)
    del features, api

benchmark = runpy.run_path(str(ROOT / "scripts/partition_benchmark.py"))
if "group_count" not in inspect.signature(benchmark["create_context"]).parameters:
    raise RuntimeError("Podmień partition_benchmark.py na wersję z trzema układami A/B/C")
ctx = benchmark["create_context"](
    root=ROOT, repeats=REPEATS, warmups=WARMUPS, threads=THREADS,
    write_statistics=WRITE_STATISTICS, row_group_size=ROW_GROUP_SIZE,
    group_count=GROUP_COUNT, reader_mode=READER_MODE,
    small_difference_percent=SMALL_DIFFERENCE_PERCENT,
)
print("Dane eksperymentu:", ctx["data"])
print("Raport:", ctx["report"])

## 3. Zapis A/B/C

A: jeden plik. B: zrównoważone grupy stacji w katalogach `station_group=.../`.
C: dotychczasowe katalogi `kod_stacji=.../`.

Wspólne: pełny zbiór danych, Snappy, writer, limit grup wierszy, porządek stacja–czas.
W B dodajemy techniczny klucz katalogu; nie jest on cechą modelu i nie wchodzi do logicznego
porównania kolumn. Mapowanie stacji na grupy jest zapisane w raporcie.

In [ ]:
storage = benchmark["write_variants"](ctx)
print(storage["tree"])
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=180):
    display(storage["storage"])
    display(storage["files"])
    display(pl.read_csv(ctx["report"] / "station_groups.csv"))
print("Liczba grup B:", ctx["actual_group_count"])
print("Liczba partycji stacji C:", len(ctx["station_groups"]))

## 4. Kontrola danych

Sprawdzamy pełne schematy logiczne, wartości i klucz stacja–czas.
Dodatkowo kontrolujemy, czy każdy wiersz B znajduje się w grupie przypisanej do jego stacji.
Kolumny technicznej `station_group` nie porównujemy z oryginalnymi cechami.

In [ ]:
validation = benchmark["verify_data"](ctx)
print(validation)
if validation["checked_columns"] != len(pl.read_parquet_schema(SOURCE)):
    raise AssertionError("Nie porównano wszystkich oryginalnych kolumn")

## 5. Scenariusze i filtry

1, 8 i 32 stacje, o ile są rzeczywistymi podzbiorami źródła, oraz wszystkie stacje.
Każdy scenariusz ma pięć konfiguracji.

- A i C: `WHERE kod_stacji IN (...)`.
- B: `WHERE station_group IN (...) AND kod_stacji IN (...)`.
- B/C bez filtra: brak `WHERE`; agregacja całego zbioru.

Filtr grup w B wynika z mapowania, nie zmienia wybranych stacji.
Listy readerów zawierają wszystkie pliki. Plan pokazuje faktycznie pomijane partycje.

In [ ]:
scenarios = benchmark["prepare_scenarios"](ctx)
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(scenarios)
for scenario in ctx["scenarios"]:
    print("\nScenariusz:", scenario["scenario"])
    for variant in ctx["variants"]:
        print("\n", benchmark["LABELS"][variant])
        print(ctx["queries"][scenario["id"], variant])

In [ ]:
# Plany filtrów B i C. Plany pozostałych konfiguracji są zapisane w raporcie.
for scenario in ctx["scenarios"]:
    if scenario["id"] == "control":
        continue
    print("\nScenariusz:", scenario["scenario"])
    for variant in ["B", "C"]:
        print("\nWariant:", variant)
        print((ctx["report"] / f"plan_{scenario['id']}_{variant}.txt").read_text(encoding="utf-8"))

## 6. Benchmark

3 rozgrzewki i 20 powtórzeń. Kolejność scenariuszy i pięciu konfiguracji jest losowana.
Dla czterech scenariuszy otrzymujemy **400 pomiarów**.
Walidacja wyników jest poza timerem; pomiar obejmuje `execute + fetchall`.
Cache systemowy nie jest czyszczony. Pełne agregacje B/C są ponawiane w każdym scenariuszu.

In [ ]:
timings = benchmark["run_benchmark"](ctx)
summary = benchmark["summarize"](ctx, timings)
expected = len(ctx["scenarios"]) * len(benchmark["CONFIGS"]) * ctx["repeats"]
if timings.height != expected:
    raise AssertionError(f"Oczekiwano {expected} pomiarów, jest {timings.height}")
print("Liczba pomiarów:", timings.height)
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=220):
    display(summary)
    display(timings.sort(["id", "iteration", "variant"]))

## 7. Porównania

Zysk = `(czas bazowy − czas badanego wariantu) / czas bazowy × 100%`.

Porównujemy tylko ten sam zakres danych:
- B i C względem A — te same wybrane stacje;
- B względem C — wpływ liczby i układu partycji dla tych samych stacji;
- B bez filtra względem C bez filtra — pełny zbiór;
- w scenariuszu wszystkich stacji także pełne agregacje względem A.

Nie obliczamy pozornego „zysku” pełnej agregacji względem A ograniczonego do jednej stacji.
Próg małej różnicy jest regułą opisową, nie testem istotności statystycznej.

In [ ]:
comparisons = pl.read_csv(ctx["report"] / "comparisons.csv")
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=220):
    display(comparisons)

## 8. Wykresy

Pierwszy wykres pokazuje wszystkie pięć konfiguracji, z zaznaczeniem różnego zakresu pracy.
Wykres zysków rozdziela zapytania dla tych samych stacji i pełne agregacje.
Ostatni wykres pokazuje wszystkie powtórzenia.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    if exc.name != "matplotlib":
        raise
    print("Brak matplotlib w środowisku kernela. Tabele i CSV są zapisane.")
else:
    figures = []
    try:
        with plt.ioff():
            figures = benchmark["make_plots"](ctx, timings, summary)
    finally:
        for figure in figures:
            plt.close(figure)

for filename, title in [
    ("01_mean_times.png", "Średnie czasy pięciu konfiguracji"),
    ("02_time_saved.png", "Zyski dla porównywalnego zakresu danych"),
    ("03_repetitions.png", "Wszystkie powtórzenia"),
]:
    path = ctx["report"] / filename
    if path.is_file():
        display(Markdown("### " + title))
        display(Image(filename=str(path), width=1200))
    else:
        print("Brak wykresu:", filename)

## 9. Wnioski

Opis wynika z aktualnych pomiarów. Rozróżnia poprawę, pogorszenie i niewielką lub niestabilną
różnicę. Osobno omawia małą liczbę grup B, podział po stacji C i odczyt bez filtra.
Nie zakłada z góry, że B będzie szybsze, a mały zbiór traktuje jako możliwą, nie udowodnioną
przyczynę strat. Raport nie interpretuje czasu jako liczby bajtów odczytanych z dysku.

In [ ]:
interpretation = benchmark["describe_results"](ctx, summary, storage)
display(Markdown(interpretation))
print("Raport:", ctx["report"])
print("Dane B:", ctx["grouped"])
print("Dane C:", ctx["partitioned"])

In [ ]:
ctx["con"].close()
print("Połączenie zamknięte. Dane źródłowe SILVER pozostają bez zmian.")